# Time-Dependent Coupling Depth Subduction Zone Setup

## Preamble

Let's start by adding the path to the modules in the `python` folder to the system path (so we can find the our custom modules).

In [ ]:
import sys, os
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)
sys.path.insert(0, os.path.join(basedir, os.path.pardir, os.path.pardir, 'python'))

Let's also load the module generated by the previous notebooks to get access to the parameters and functions defined there.

In [ ]:
from fenics_sz.sz_problems.sz_problem import StokesSolverNest, TemperatureSolver
from fenics_sz.sz_problems.sz_tdep_dislcreep import TDDislSubductionProblem

Then let's load all the required modules at the beginning.

In [ ]:
import dolfinx as df
import numpy as np
import pathlib
output_folder = pathlib.Path(os.path.join(basedir, "output"))
output_folder.mkdir(exist_ok=True, parents=True)

In [ ]:
class TDCDDislSubductionProblem(TDDislSubductionProblem):
    def members(self):
        # initialize the standard members in the parent class
        super().members()

        # add additional parameter for time-dependent coupling
        self.allowed_input_parameters += ["cd0", "cdf", "dcd", "tc0", "tcf"]
        self.required_parameters += ["cd0", "cdf", "dcd", "tc0", "tcf"]
        self.required_parameters += ["As"] # this was previously only required if oceanic

        self.cd0 = None # initial coupling depth
        self.cdf = None # final coupling depth
        self.dcd = None # partial coupling depth offset
        self.tc0 = None # time for initial coupling (in Myr)
        self.tcf = None # time for full coupling (in Myr)

        self.t_Myr = 0.0 # current dimensional time (in Myr)
        self.tdep_bcs = ['vw_slabtop'] # one time-dependet boundary condition

    # Overload the vw_slabtop method
    def vw_slabtop(self, x):
        """
        Return the wedge velocity on the slab surface
        """
        # work out the current coupling depth from the time
        # FIXME: this really should be stored
        cd = min(max(self.cd0, self.cd0 + (self.cdf - self.cd0)/(self.tcf - self.tc0)*(self.t_Myr - self.tc0)), self.cdf)
        pcd = cd - self.dcd # current partial coupling depth
        v = np.empty((self.gdim, x.shape[1]))
        for i in range(x.shape[1]):
            v[:,i] = min(max(-(x[1,i]+pcd)/self.dcd, 0.0), 1.0)*self.Vs_nd*self.geom.slab_spline.unittangentx(x[0,i])
        return v